# Preprocessing 2
This is a revised preprocessing file. It will perform multiple tasks required to improve the raw dataset file exported with the ConflictCalc console app into a ML ready parquet file. 

## Actions:
* Merge in country fact columns
* Merge in election dates data
* Column type conversion
  * Country is converted to categorical, this is with a fixed list of countries (10 in total) and which is exported as a .csv file for reuse. 
* Column renaming to remove special characters
* Deletion of the final, likely incomplete period
* Application of the new Id column
* Application of month_sin and month_cos columns
* Deletion of the periodEnd column
* Deletion of values for which there is not a consistent number of observations within the series (id)

## Not applied in this notebook:
* Application of the TBG features
    * Not relevant for the Chronos 2 model: avoids creating features only to have to delete them given memory usage has been an issue. 
* Separation into training and testing sets
    * As above
* Application of target variables
    * May change between notebooks: TGB model targets are exected to be "conflict at any point between t and t+n." Chronos 2 behaves differently and outputs the probabilities of the full range of targets (i.e. 12 periods) so that can be derived from the output. 

In [1]:
import pandas as pd
df = pd.read_csv("../Data/Dataset Original/eastafrica3_20150101_4c90a061-413b-4eb8-90b5-84413420100b.csv")
#df.to_parquet('../Data/EastAfrica Preprocessed/eastafrica.parquet')

In [2]:
print("Rows x columns:", df.shape)

Rows x columns: (2331560, 38)


In [3]:
df.head(5)

,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,LocalDistinctActorCount,...,RegionalMeanSeverity,RegionalMaxSeverity,RegionalProtestersEventCount,RegionalStateForcesEventCount,RegionalPoliticalMilitiaEventCount,RegionalIdentityMilitiaEventCount,RegionalCiviliansEventCount,RegionalRebelGroupEventCount,RegionalRiotersEventCount,RegionalOtherTypeEventCount
0,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01T00:00:00,2015-01-29T00:00:00,3,5,...,2.000000,2,0,0,1,1,2,0,0,0
1,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01T00:00:00,2015-01-29T00:00:00,0,0,...,2.800000,4,0,2,3,1,3,1,0,0
2,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01T00:00:00,2015-01-29T00:00:00,0,0,...,2.666667,4,0,2,4,1,4,1,0,0
3,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01T00:00:00,2015-01-29T00:00:00,0,0,...,2.000000,2,0,0,1,0,1,0,0,0
4,Kitagata,Uganda,-0.6736,30.1540,48.589624,290.160533,2015-01-01T00:00:00,2015-01-29T00:00:00,0,0,...,0.000000,0,0,0,0,0,0,0,0,0


In [4]:
unique_countries = df["country"].unique()
print(unique_countries)
print("Countries: ", len(unique_countries))

unique_years = sorted(df['periodStart'].str.slice(0, 4).unique())
print(unique_years)
print("Years: ", unique_years)

['South Sudan' 'Uganda' 'Sudan' 'Ethiopia' 'Somalia' 'Kenya' 'Libya'
 'Central African Republic' 'Egypt' 'Chad']
Countries:  10
['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Years:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


## Delete Final Period
As it is not a full period of 28 days. 

In [5]:
max_period_start = max(df['periodStart'])
df = df[df['periodStart'] < max_period_start]
print(f"Deleted max period {max_period_start}")



Deleted max period 2025-08-28T00:00:00


In [6]:
print("Overall periodStart range: ", min(df['periodStart']), " - ", max(df['periodStart']))
print("Periods:", len(df["periodStart"].unique()))


Overall periodStart range:  2015-01-01T00:00:00  -  2025-07-31T00:00:00
Periods: 139


## Apply the Id column

In [7]:
df['id'] = (df['country'].str.replace(" ", "_")
 + "_" + df['name']).str.replace(" ", "_")

cols = ['id'] + [c for c in df.columns if c != 'id']
df = df[cols]

df.head()

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,RegionalMeanSeverity,RegionalMaxSeverity,RegionalProtestersEventCount,RegionalStateForcesEventCount,RegionalPoliticalMilitiaEventCount,RegionalIdentityMilitiaEventCount,RegionalCiviliansEventCount,RegionalRebelGroupEventCount,RegionalRiotersEventCount,RegionalOtherTypeEventCount
0,South_Sudan_Lakes,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01T00:00:00,2015-01-29T00:00:00,3,...,2.000000,2,0,0,1,1,2,0,0,0
1,South_Sudan_Rumbek_East,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01T00:00:00,2015-01-29T00:00:00,0,...,2.800000,4,0,2,3,1,3,1,0,0
2,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01T00:00:00,2015-01-29T00:00:00,0,...,2.666667,4,0,2,4,1,4,1,0,0
3,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01T00:00:00,2015-01-29T00:00:00,0,...,2.000000,2,0,0,1,0,1,0,0,0
4,Uganda_Kitagata,Kitagata,Uganda,-0.6736,30.1540,48.589624,290.160533,2015-01-01T00:00:00,2015-01-29T00:00:00,0,...,0.000000,0,0,0,0,0,0,0,0,0


### Unique Ids

In [8]:
unique_ids = df["id"].unique()
print("Unique Ids: ", len(unique_ids))

Unique Ids:  16642


## Column Type Conversion

### Numerics Conversation

In [9]:
## Numeric Conversaion, also sets any missing values to NaN
df['latitude'] = pd.to_numeric(df['latitude'], errors = 'coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors = 'coerce')

### Date Conversion

In [10]:
df['periodStart'] = pd.to_datetime(df['periodStart'], format ='%Y-%m-%dT%H:%M:%S')
df['periodEnd'] = pd.to_datetime(df['periodEnd'], format ='%Y-%m-%dT%H:%M:%S')

### Categorical Conversion
Categorical conversion for a handful of fields. Note that that we need to save the mappings of the original strings into a new csv file so that we can reuse the mapping in ConflictQuery

In [11]:
df['id'] = df["id"].astype("category")
df['name'] = df["name"].astype("category")
df['country'] = df["country"].astype("category")

In [12]:
### Confirm types:

In [13]:
for col in df.columns:
    print (f"{col}: ", df[col].dtype)

id:  category
name:  category
country:  category
latitude:  float64
longitude:  float64
minBorderDistanceKm:  float64
minCapitalDistanceKm:  float64
periodStart:  datetime64[ns]
periodEnd:  datetime64[ns]
LocalEventCount:  int64
LocalDistinctActorCount:  int64
LocalDistinctEventTypes:  int64
LocalDistinctEventSubtypes:  int64
LocalTotalFatalities:  int64
LocalMeanSeverity:  float64
LocalMaxSeverity:  int64
LocalProtestersEventCount:  int64
LocalStateForcesEventCount:  int64
LocalPoliticalMilitiaEventCount:  int64
LocalIdentityMilitiaEventCount:  int64
LocalRebelGroupEventCount:  int64
LocalRiotersEventCount:  int64
LocalCiviliansEventCount:  int64
LocalOtherTypeEventCount:  int64
RegionalEventCount:  int64
RegionalDistinctActorCount:  int64
RegionalDistinctEventTypes:  int64
RegionalDistinctEventSubTypes:  int64
RegionalTotalFatalities:  int64
RegionalMeanSeverity:  float64
RegionalMaxSeverity:  int64
RegionalProtestersEventCount:  int64
RegionalStateForcesEventCount:  int64
RegionalPo

## Apply the Month Columns and Delete 
monthSin and monthCos should give a proper reflection of the month the period mostly takes place in, while keeping December and January close together (i.e. solving the issue of December = 11, January = 1)

In [14]:
import numpy as np
df["period_midpoint"] = df["periodStart"] + (df["periodEnd"] - df["periodStart"]) / 2

df["month"] = df["period_midpoint"].dt.month #temporary
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df = df[df.columns.drop(['month'])] #Remove the raw month: we don't care since the sin and cos variations capture thsi better.
df = df[df.columns.drop(['period_midpoint'])] #Remove the period_midpoint: no longer required.

### Export Country categorical values to a separate datasource
This is required when looking up data in ConflictQuery.

In [15]:
#Deterministic category codes. 
countries = [
'Central African Republic',
'Chad',
'Egypt',
'Ethiopia',    
'Kenya',
'Libya',
'South Sudan', 
'Sudan',
'Somalia',
'Uganda'   
]

df['country'] = pd.Categorical(df['country'], categories = countries)

df_country_categories = pd.DataFrame({
    "country": df['country'].cat.categories,
    "code" : range(len(df['country'].cat.categories))
})
df_country_categories

,country,code
0,Central African Republic,0
1,Chad,1
2,Egypt,2
3,Ethiopia,3
4,Kenya,4
5,Libya,5
6,South Sudan,6
7,Sudan,7
8,Somalia,8
9,Uganda,9


In [16]:
df_country_categories.to_csv("../Data/EastAfrica Preprocessed/eastafrica_country_categorical_codes.csv", mode = "w")

## Identify training and testing periods
Note that the last 12 periods are taken from the final of the dataset, at this point they are a guide only as the future targets may be null if there is not enough data present. Likewise, lag features may cause the first n periods to have null values in TGB models, so those might be deleted too. Each modelling notebook will apply it's own cleaning for these. 

In [17]:
all_period_starts = sorted(df['periodStart'].unique(), reverse = True)
all_period_starts
testing_periods = all_period_starts[:12]
testing_periods

training_periods = [v for v in all_period_starts if v not in testing_periods]
training_periods

print("Training periods: ", min(training_periods), " - ",max(training_periods))
print("Testing periods: ", min(testing_periods), " - ", max(testing_periods))

Training periods:  2015-01-01 00:00:00  -  2024-08-29 00:00:00
Testing periods:  2024-09-26 00:00:00  -  2025-07-31 00:00:00


## Merging Country Facts Data

In [18]:
df_country_facts = pd.read_csv("../Data/Country Data Preprocessed/Country Facts.csv")
df_country_facts.head(5)

,Country Name,Country Code,Year,Time Code,"Population, total [SP.POP.TOTL]",Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST],Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST],"Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]",Children out of school (% of primary school age) [SE.PRM.UNER.ZS],GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD],...,GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag,"Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag",Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag,Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag,"Population, total [SP.POP.TOTL]_missing_flag","Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag",Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag,Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag"
0,Central African Republic,NaN,1990,YR1990,2871910,-1.260176,-1.226337,32.014149,41.592239,387.280283,...,0,1,1,1,0,0,0,0,1,0
1,Central African Republic,NaN,2000,YR2000,3833416,-1.260176,-1.226337,32.014149,41.592239,387.280283,...,0,1,1,1,0,0,1,0,0,0
2,Central African Republic,NaN,2016,YR2016,4713663,-1.308185,-1.688903,38.480461,41.592239,387.280283,...,0,0,0,0,0,0,0,0,0,0
3,Central African Republic,NaN,2017,YR2017,4793511,-1.180782,-1.605281,38.480461,41.592239,408.750638,...,0,0,0,0,0,0,1,0,0,0
4,Central African Republic,NaN,2018,YR2018,4878657,-1.220890,-1.553388,38.480461,41.592239,509.122487,...,0,0,0,0,0,0,1,0,0,0


In [19]:
df_country_facts["Year"] =  df_country_facts['Year'].astype(int)
df_country_facts['Country Name'] = pd.Categorical(df_country_facts['Country Name'], categories = countries) #reuse the countries collection of unique countires, try and avoid country column ending up as object dtype in the merge. 

#Drop data older than 2015, it's added by the World Bank Dataset tool autmatically. 
df_country_facts_rows = df_country_facts[df_country_facts["Year"] < 2015].index
df_country_facts.drop(df_country_facts_rows, inplace=True)

#Drop Time Code and Country Code columns: not required. 
df_country_facts = df_country_facts.drop(["Time Code", "Country Code"], axis=1)

df_country_facts.head()

,Country Name,Year,"Population, total [SP.POP.TOTL]",Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST],Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST],"Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]",Children out of school (% of primary school age) [SE.PRM.UNER.ZS],GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD],GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD],Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS],...,GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag,"Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag",Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag,Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag,"Population, total [SP.POP.TOTL]_missing_flag","Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag",Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag,Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag"
2,Central African Republic,2016,4713663,-1.308185,-1.688903,38.480461,41.592239,387.280283,376.858254,0.461758,...,0,0,0,0,0,0,0,0,0,0
3,Central African Republic,2017,4793511,-1.180782,-1.605281,38.480461,41.592239,408.750638,387.357951,0.445899,...,0,0,0,0,0,0,1,0,0,0
4,Central African Republic,2018,4878657,-1.220890,-1.553388,38.480461,41.592239,509.122487,395.020015,0.556067,...,0,0,0,0,0,0,1,0,0,0
5,Central African Republic,2019,4944703,-1.201425,-1.583995,38.480461,41.592239,518.991035,401.825822,0.540285,...,0,0,0,0,0,0,1,0,0,0
6,Central African Republic,2020,5026628,-1.252405,-1.562382,38.480461,41.592239,506.664683,398.834274,0.535439,...,0,0,0,0,0,0,1,0,0,0


In [20]:
# Add the raw features (can add the imputation flags if required)
df_country_facts_feature_columns = [
    'Population, total [SP.POP.TOTL]',
    'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]',
    'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]',
    'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]',
    'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]',
    'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]',
    'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]',
    'Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]',
    'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]',
    'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]',
    'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]',
    'Permanent cropland (% of land area) [AG.LND.CROP.ZS]',
    'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]',
    'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]'
]

facts_subset = df_country_facts[["Country Name", "Year"] + df_country_facts_feature_columns]


#Context data:
df["year"] = pd.to_datetime(df["periodStart"]).dt.year
df = df.merge(
    facts_subset,
    left_on=["country", "year"],
    right_on=["Country Name", "Year"],
    how="left"
)


In [21]:
#Drop the temp columns from df:
df = df.drop(["year"], axis=1)
df = df.drop(["Year"], axis=1)
df = df.drop(["Country Name"], axis=1)


In [22]:
del df_country_facts

In [23]:
df.head(5)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,Children out of school (% of primary school age) [SE.PRM.UNER.ZS],GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD],GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD],Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS],Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS],Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC],Cereal yield (kg per hectare) [AG.YLD.CREL.KG],Permanent cropland (% of land area) [AG.LND.CROP.ZS],"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]","Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]"
0,South_Sudan_Lakes,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01,2015-01-29,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,South_Sudan_Rumbek_East,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Uganda_Kitagata,Kitagata,Uganda,-0.6736,30.1540,48.589624,290.160533,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
#for col in df.columns:
   # print (f"{col}: ", df[col].dtype)

### Merging Election Data 
Also drops periodEnd: no longer required once this process completes. 

In [25]:
df_elections = pd.read_csv("../Data/Election Data Preprocessed/Elections.csv")
df_elections["Date"] = pd.to_datetime(df_elections["Date"], format="%d/%m/%Y")
df_elections['Country'] = pd.Categorical(df_elections['Country'], categories = countries) # again reusing the fixed list of countries. 
df_elections.head()

,Country,Date,Type
0,Chad,2016-04-10,PRESIDENTIAL ELECTION
1,Chad,2021-04-11,PRESIDENTIAL ELECTION
2,Egypt,2011-09-28,LEGISLATIVE ELECTION
3,Egypt,2011-09-29,LEGISLATIVE ELECTION
4,Egypt,2015-10-17,LEGISLATIVE ELECTION


In [26]:
#Unique periods by country
tmp = df[['country', 'periodStart', 'periodEnd']].drop_duplicates()

#merge elections into tmp
tmp = tmp.merge(
    df_elections[['Country', 'Date']],
    how='left',
    left_on='country',
    right_on='Country'
)

#create is_voting before grouping
tmp['is_voting'] = (
    (tmp['Date'] >= tmp['periodStart']) &
    (tmp['Date'] <= tmp['periodEnd'])
).astype(int)

#Collapse multiple elections per period
tmp = (
    tmp.groupby(['country', 'periodStart', 'periodEnd'], observed = False)['is_voting']
       .max()
       .reset_index()
)

#merge back to df
df = df.merge(
    tmp,
    how='left',
    on=['country', 'periodStart', 'periodEnd']
)


df['is_voting'] = df['is_voting'].astype(bool) #Ensure Is_voting is bool

df = df[df.columns.drop(['periodEnd'])] #Drop periodEnd: no longer required. 

del tmp
del df_elections

C:\Users\andre\AppData\Local\Temp\ipykernel_15048\480466341.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby(['country', 'periodStart', 'periodEnd'])['is_voting']


In [27]:
df.head(5)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,LocalEventCount,LocalDistinctActorCount,...,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD],GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD],Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS],Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS],Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC],Cereal yield (kg per hectare) [AG.YLD.CREL.KG],Permanent cropland (% of land area) [AG.LND.CROP.ZS],"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]","Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]",is_voting
0,South_Sudan_Lakes,Lakes,South Sudan,6.8074,29.6760,235.565742,302.791866,2015-01-01,3,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,South_Sudan_Rumbek_East,Rumbek East,South Sudan,6.6757,29.8111,222.810158,281.869229,2015-01-01,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,Uganda_Kitagata,Kitagata,Uganda,-0.6736,30.1540,48.589624,290.160533,2015-01-01,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [28]:
for col in df.columns:
    print (f"{col}: ", df[col].dtype)

id:  category
name:  category
country:  category
latitude:  float64
longitude:  float64
minBorderDistanceKm:  float64
minCapitalDistanceKm:  float64
periodStart:  datetime64[ns]
LocalEventCount:  int64
LocalDistinctActorCount:  int64
LocalDistinctEventTypes:  int64
LocalDistinctEventSubtypes:  int64
LocalTotalFatalities:  int64
LocalMeanSeverity:  float64
LocalMaxSeverity:  int64
LocalProtestersEventCount:  int64
LocalStateForcesEventCount:  int64
LocalPoliticalMilitiaEventCount:  int64
LocalIdentityMilitiaEventCount:  int64
LocalRebelGroupEventCount:  int64
LocalRiotersEventCount:  int64
LocalCiviliansEventCount:  int64
LocalOtherTypeEventCount:  int64
RegionalEventCount:  int64
RegionalDistinctActorCount:  int64
RegionalDistinctEventTypes:  int64
RegionalDistinctEventSubTypes:  int64
RegionalTotalFatalities:  int64
RegionalMeanSeverity:  float64
RegionalMaxSeverity:  int64
RegionalProtestersEventCount:  int64
RegionalStateForcesEventCount:  int64
RegionalPoliticalMilitiaEventCount:  

## Renaming All Columns
Removing special characters is required by the LightGBM model

In [29]:
import re
new_cols = {}

for col in df.columns:
    new_name = col.replace(" ", "_") #Replace spaces with underscores
    new_name = re.sub(r"[^A-Za-z0-9_+]", "", new_name) #remove any character that is not A-Z, a-z, 0-9, or underscore or +. + included as it's already in the target_columns. 
    new_cols[col] = new_name

df = df.rename(columns=new_cols)

## Outputting column names and types

In [30]:
for col in df.columns:
    print (f"{col}: ", df[col].dtype)

id:  category
name:  category
country:  category
latitude:  float64
longitude:  float64
minBorderDistanceKm:  float64
minCapitalDistanceKm:  float64
periodStart:  datetime64[ns]
LocalEventCount:  int64
LocalDistinctActorCount:  int64
LocalDistinctEventTypes:  int64
LocalDistinctEventSubtypes:  int64
LocalTotalFatalities:  int64
LocalMeanSeverity:  float64
LocalMaxSeverity:  int64
LocalProtestersEventCount:  int64
LocalStateForcesEventCount:  int64
LocalPoliticalMilitiaEventCount:  int64
LocalIdentityMilitiaEventCount:  int64
LocalRebelGroupEventCount:  int64
LocalRiotersEventCount:  int64
LocalCiviliansEventCount:  int64
LocalOtherTypeEventCount:  int64
RegionalEventCount:  int64
RegionalDistinctActorCount:  int64
RegionalDistinctEventTypes:  int64
RegionalDistinctEventSubTypes:  int64
RegionalTotalFatalities:  int64
RegionalMeanSeverity:  float64
RegionalMaxSeverity:  int64
RegionalProtestersEventCount:  int64
RegionalStateForcesEventCount:  int64
RegionalPoliticalMilitiaEventCount:  

## Validation of Series (Ids) length

In [41]:

counts = (
    df.groupby("id", observed = True)
      .size()
      .reset_index(name="count")
)

bad_ids = counts.query("count != 139")

print("Bad IDs in df:", bad_ids["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df):")
print(bad_ids)


Bad IDs in df: []
Counts for bad IDs (df):
Empty DataFrame
Columns: [id, count]
Index: []


### Delete "Bad" Ids
These have probably been created through shared place names. There are 5 in total, it looks like they are created from duplicated place names somewhere. Delete them to avoid them affecting future ML. 


In [38]:
df = df[~df["id"].isin(bad_ids["id"].tolist())]

In [40]:
#validate again, should be none now.
counts = (
    df.groupby("id", observed = True)
      .size()
      .reset_index(name="count")
)

bad_ids = counts.query("count != 139")

print("Bad IDs in df:", bad_ids["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df):")
print(bad_ids)


Bad IDs in df: []
Counts for bad IDs (df):
Empty DataFrame
Columns: [id, count]
Index: []


## Save Final Dataset

In [31]:
df = df.sort_values(by=["periodStart", "id"], ascending=True) #sort by periodStart, then id.
df.to_parquet('../Data/EastAfrica Preprocessed/eastafrica.parquet')
print("Rows x columns: ", df.shape)

Rows x columns:  (2314906, 55)
